# M1-lite training notebook

**Project J26-IT-423 · Component M3 (Explainable AI & Decision Support) · IT23293076**

M1's real hybrid LSTM + XGBoost forecaster is not finished, so M3 is built against **M1-lite**: a light, clearly labelled stand-in trained on M1's own dataset. This notebook trains and evaluates it step by step:

1. Load M1's raw monthly dataset
2. Clean it (units, duplicates, data-entry errors) with a full log
3. Build lag-safe features for a one-month-ahead forecast
4. Walk-forward validation (2019-2021) to choose the blend weight and the 80% interval
5. Held-out test (2022-2024) against simple baselines
6. Fit and save the deployed demo model
7. Check that it supports M3: SHAP additivity, driver groups, what-if scenarios

The notebook uses the same functions as `scripts/train_m1lite.py`, so both always produce the same model.

> **Note:** M1-lite is for building and demonstrating M3. Its results are not M1's results.

## 0. Setup

If Windows *Controlled Folder Access* blocks Python from saving under `Documents`, uncomment the `M3_OUT_DIR` line to save outputs somewhere else (or allow `python.exe` in Windows Security).

In [ ]:
import os
import sys
import time
from pathlib import Path

# os.environ["M3_OUT_DIR"] = r"C:\m3_out"   # optional: where models/ and data/processed/ are written

# Find the project folder (the one containing tea_m3/) from wherever the notebook runs.
ROOT = Path(os.environ.get("M3_ROOT") or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tea_m3").is_dir()))
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from tea_m3 import config
from tea_m3.data_prep import clean, load_raw
from tea_m3.features import build_features, fit_rain_climatology
from tea_m3.forecasting import ScenarioError
from tea_m3.forecasting.m1lite import INTERVAL_LEVEL, XGB_PARAMS, M1Lite, fit_branches, training_rows
from train_m1lite import BLEND_GRID, metric_table, walk_forward

pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print("Project folder:", ROOT)
print("Raw data:      ", config.RAW_CSV)
print("Outputs:       ", config.OUT_DIR)
print("XGBoost params:", XGB_PARAMS)

## 1. Load M1's raw dataset

`M1_combined_monthly_raw.csv` holds 77 columns: Tea Board exports, UN Comtrade buyer volumes, Colombo auction prices, ONI (ENSO), rainfall, FX, CPI, competitors and buyer-country macro data.

In [ ]:
raw = load_raw()
print(f"{raw.shape[0]} rows x {raw.shape[1]} columns, {raw['Date'].iloc[0]} to {raw['Date'].iloc[-1]}")
raw.iloc[:5, :9]

### Why cleaning is needed

Summing the raw export totals per year (after converting kg to million kg) gives impossible years. Sri Lanka exports roughly 230-320 million kg a year.

In [ ]:
dedup = raw.drop_duplicates("Date")
raw_mkg = np.where(dedup["TB_Quantity_Unit"].eq("kg"), dedup["TB_Total_Exports"] / 1e6, dedup["TB_Total_Exports"])
raw_yearly = pd.Series(raw_mkg, index=dedup["Date"].str[:4].astype(int)).groupby(level=0).sum()

print("Duplicate months in raw file:", raw.loc[raw["Date"].duplicated(), "Date"].tolist())
print("Unit column values:", raw["TB_Quantity_Unit"].value_counts().to_dict())
raw_yearly[raw_yearly > 350].round(1).rename("raw yearly total (mn kg)")

## 2. Clean the data

`clean()` keeps only the series M1-lite and M3 need and records every correction:

| Problem | Fix |
|---|---|
| Duplicate months | Dropped |
| Units change (million kg to 2005, kg after) | Converted to million kg |
| Totals with a x10 error | Replaced by the sum of the 5 product rows |
| Months with no export figure | Interpolated **for lag features only**, never used as targets |
| Auction values inconsistent with the other elevations | Replaced from the cross-elevation ratio (training period) |
| Rainfall `-999` and impossible values | Set missing, filled by training climatology |
| ONI seasons centred on the month (future leak) | Shifted 2 months so each season is used only after it ends |
| Buyer demand | Constructed proxy: lagged GDP growth weighted by Comtrade shares |

In [ ]:
clean_df, log = clean(raw)
log_df = pd.DataFrame(log)
print(f"Clean table: {len(clean_df)} months, {clean_df.index[0]} to {clean_df.index[-1]}")
print(f"Cleaning log: {len(log_df)} entries\n")
print(log_df["field"].value_counts().to_string())
log_df.head(15)

In [ ]:
clean_yearly = clean_df["exports_mkg"].groupby(clean_df.index.year).sum()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(raw_yearly.index, raw_yearly.values, "o--", color="tab:red", label="raw file")
ax.plot(clean_yearly.index, clean_yearly.values, "o-", color="tab:blue", label="cleaned")
ax.set(title="Yearly tea exports: raw vs cleaned", ylabel="million kg")
ax.legend()
plt.show()

In [ ]:
series = {
    "exports_mkg": "Exports (million kg)",
    "price_lkr_kg": "Auction price (LKR/kg)",
    "usd_lkr": "LKR per USD",
    "rain_mm": "Rainfall, 3-station mean (mm)",
    "oni": "ONI anomaly, lag-safe (deg C)",
    "bdi": "Buyer-demand index (proxy)",
}
t = clean_df.index.to_timestamp()
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
for ax, (col, title) in zip(axes.ravel(), series.items()):
    ax.plot(t, clean_df[col], lw=1)
    ax.set_title(title, fontsize=10)
for ax in axes.ravel():
    for start, end, colour in [(config.VAL_START, config.VAL_END, "orange"), (config.TEST_START, config.TEST_END, "green")]:
        ax.axvspan(pd.Period(start, "M").to_timestamp(), pd.Period(end, "M").to_timestamp(), color=colour, alpha=0.08)
fig.suptitle("Cleaned series (orange = validation, green = test)")
fig.tight_layout()
plt.show()

print("Synthetic FX months (as labelled in M1's file):", int(clean_df["fx_synthetic"].sum()), "- confirm their origin with the M1 owner.")

## 3. Features

Each row is an **origin month t**; features use data up to t only, and the target is exports in **t + 1**.

- **Export momentum:** `exp_lag1/2/3/12`, `exp_roll3`
- **Seasonality:** `month_sin`, `month_cos` (target month)
- **Auction prices:** `price_usd_kg` (LKR price / FX), `price_chg_3m`, `price_vol_6m`
- **Exchange rate:** `fx_chg_3m`, `fx_chg_12m`
- **Rainfall:** `rain_mm`, `rain_anom` (vs training-only climatology)
- **ENSO:** `oni`, `oni_lag3`
- **Buyer demand:** `bdi`

Nominal LKR levels (FX about 77 in 2000, about 330 in 2023) are used only as USD price and % changes, because tree models cannot extrapolate beyond training levels.

In [ ]:
rain_clim = fit_rain_climatology(clean_df, config.TRAIN_END)
features = build_features(clean_df, rain_clim)
features[config.XGB_FEATURES].describe().T[["count", "mean", "std", "min", "max"]].round(3)

In [ ]:
target_period = pd.PeriodIndex(features["target_period"], freq="M")
usable = features["target"].notna() & ~features["target_imputed"] & features[config.TEMPORAL_FEATURES].notna().all(axis=1)
splits = {
    "train": (str(target_period.min()), config.TRAIN_END),
    "validation": (config.VAL_START, config.VAL_END),
    "test": (config.TEST_START, config.TEST_END),
}
pd.DataFrame({
    name: {"target months": f"{a} to {b}",
           "usable rows": int((usable & (target_period >= pd.Period(a, "M")) & (target_period <= pd.Period(b, "M"))).sum())}
    for name, (a, b) in splits.items()
}).T

## 4. Model and walk-forward validation

M1-lite has two branches with a **fixed** blend weight *w*:

$$\text{forecast} = (1-w)\cdot\text{XGBoost} + w\cdot\text{temporal}$$

- **XGBoost branch:** 100 shallow trees on all 17 features (the part SHAP explains)
- **Temporal branch:** linear seasonal-autoregressive model, standing in for M1's LSTM

**Walk-forward:** for every validation month, both branches are refitted on all earlier months, then forecast one month ahead. *w* and the 80% interval are chosen from validation forecasts only, never from test months.

In [ ]:
t0 = time.perf_counter()
val = walk_forward(features, config.VAL_START, config.VAL_END)
print(f"Validation walk-forward: {len(val)} months in {time.perf_counter() - t0:.1f} s")

val_mae = {w: float(((1 - w) * val["xgb"] + w * val["temporal"] - val["actual"]).abs().mean()) for w in BLEND_GRID}
w = min(val_mae, key=val_mae.get)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(val_mae), list(val_mae.values()), "o-")
ax.axvline(w, color="tab:red", ls="--", label=f"chosen w = {w}")
ax.set(title="Validation MAE by blend weight", xlabel="w (weight on temporal branch)", ylabel="MAE (million kg)")
ax.legend()
plt.show()
print(f"Chosen blend weight w = {w} (validation MAE {val_mae[w]:.3f} million kg)")

In [ ]:
val_resid = val["actual"] - ((1 - w) * val["xgb"] + w * val["temporal"])
tail = (1 - INTERVAL_LEVEL) / 2
q_lo, q_hi = float(val_resid.quantile(tail)), float(val_resid.quantile(1 - tail))
print(f"80% interval = forecast + [{q_lo:.3f}, {q_hi:.3f}] million kg (empirical validation residuals)")

metric_table(val, w).round(3)

## 5. Held-out test (2022-2024)

Same walk-forward procedure with *w* fixed. Baselines on identical months:

- **Seasonal naive:** same month last year
- **Persistence:** last month's exports

Theil's U < 1 means a lower RMSE than persistence.

In [ ]:
t0 = time.perf_counter()
test = walk_forward(features, config.TEST_START, config.TEST_END)
test["blend"] = (1 - w) * test["xgb"] + w * test["temporal"]
test["lower"], test["upper"] = test["blend"] + q_lo, test["blend"] + q_hi
coverage = float(((test["actual"] >= test["lower"]) & (test["actual"] <= test["upper"])).mean())
print(f"Test walk-forward: {len(test)} months in {time.perf_counter() - t0:.1f} s")
print(f"80% interval coverage on test: {coverage:.0%} (target 80%)")

test_table = metric_table(test, w)
test_table.round(3)

In [ ]:
tt = pd.PeriodIndex(test["target_period"], freq="M").to_timestamp()
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(tt, test["lower"], test["upper"], color="tab:blue", alpha=0.15, label="80% interval")
ax.plot(tt, test["actual"], "k-o", ms=3, label="actual")
ax.plot(tt, test["blend"], "-", color="tab:blue", label="M1-lite forecast")
ax.plot(tt, test["seasonal_naive"], ":", color="tab:gray", label="seasonal naive")
ax.set(title="Held-out test: one-month-ahead forecasts, 2022-2024", ylabel="exports (million kg)")
ax.legend(ncol=4, fontsize=8)
plt.show()

**How to read this honestly:** M1-lite clearly beats the seasonal-naive baseline but only roughly matches persistence over 2022-2024, a period that includes Sri Lanka's economic crisis. That is enough to build and demonstrate M3. It is not evidence that the forecasting problem is solved.

## 6. Fit and save the deployed demo model

The deployed model is fitted on all targets up to **2021-12**, so every 2022-2024 month shown in the M3 demo is genuinely held out.

In [ ]:
import hashlib

data_hash = hashlib.sha256(Path(config.RAW_CSV).read_bytes()).hexdigest()
xgb, temporal = fit_branches(training_rows(features, config.VAL_END))
summary = {"blend_weight": w, "interval_80_residual_quantiles": [q_lo, q_hi],
           "test_interval_80_coverage": coverage, "test": test_table.to_dict(orient="index")}
model = M1Lite(xgb, temporal, w, clean_df, rain_clim, train_end=config.VAL_END, data_hash=data_hash,
               residual_quantiles=(q_lo, q_hi), metrics=summary)

try:
    path = model.save()
    print(f"Saved: {path} ({path.stat().st_size / 1024:.0f} KB)")
except OSError as err:
    path = None
    print("Could not save the model:", err)
    print("Windows Controlled Folder Access may be blocking Python. Allow python.exe, or set M3_OUT_DIR in the setup cell.")

model.model_version()

## 7. Check that it supports M3

### 7.1 SHAP additivity for one held-out month

With a fixed *w*, the forecast splits exactly into the explained XGBoost part and one separate temporal term:

$$\text{forecast} = (1-w)\,\text{base} + \sum_i (1-w)\,\phi_i + w\cdot\text{temporal}$$

In [ ]:
ORIGIN = "2024-03"
r = model.forecast(ORIGIN)
explainer = shap.TreeExplainer(model.get_xgb_model())
x = pd.DataFrame([r["features_used"]], columns=model.feature_names()).astype(float)
phi = explainer.shap_values(x)[0]
base = float(np.ravel(explainer.expected_value)[0])
w_ = r["blend_weight"]

reconstructed = (1 - w_) * base + ((1 - w_) * phi).sum() + w_ * r["temporal_pred"]
print(f"Forecast for {r['target_period']}: {r['forecast_value']:.3f} million kg "
      f"(80% {r['interval_lower']:.2f} to {r['interval_upper']:.2f}); actual {r['actual_value']:.3f}")
print(f"Reconstructed from SHAP: {reconstructed:.6f}   difference {abs(reconstructed - r['forecast_value']):.2e}")
assert abs(reconstructed - r["forecast_value"]) < 1e-4, "SHAP decomposition does not reconcile"
print("Warnings:", r["warnings"])

In [ ]:
contrib = dict(zip(model.feature_names(), (1 - w_) * phi))
groups = pd.Series({g: sum(contrib[f] for f in feats) for g, feats in config.DRIVER_GROUPS.items()}).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(groups.index, groups.values, color=["#0072B2" if v >= 0 else "#E69F00" for v in groups.values])
ax.axvline(0, color="k", lw=0.8)
ax.set(title=f"Model-attributed driver contributions, forecast for {r['target_period']}\n"
             f"(base {(1 - w_) * base:.2f} + temporal term {w_ * r['temporal_pred']:.2f} shown separately)",
       xlabel="contribution to forecast (million kg); blue = up, orange = down")
plt.show()
groups.round(3).rename("million kg")

### 7.2 Global driver importance on the test months

Mean absolute SHAP contribution per business driver over the held-out test origins. This shows what the XGBoost branch relies on overall. It describes the model, not real-world causes.

In [ ]:
test_origins = pd.PeriodIndex(test["origin"], freq="M")
X_test = features.loc[test_origins, config.XGB_FEATURES].astype(float)
phi_test = pd.DataFrame(explainer.shap_values(X_test) * (1 - w_), columns=config.XGB_FEATURES)
global_groups = pd.Series({g: phi_test[feats].sum(axis=1).abs().mean() for g, feats in config.DRIVER_GROUPS.items()}).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(global_groups.index, global_groups.values, color="#0072B2")
ax.set(title="Mean |contribution| per driver, test months 2022-2024", xlabel="million kg")
plt.show()

### 7.3 What-if scenarios

A scenario changes a raw driver at the origin month. All dependent features are then recalculated and the same model is re-run. Values outside the supported range are rejected, and unusual values give a warning.

*Model sensitivity, not a causal prediction.*

In [ ]:
ranges = model.driver_ranges(ORIGIN)
pd.DataFrame(ranges).T[["label", "unit", "current", "min", "p5", "p95", "max"]]

In [ ]:
scenario = {"rain_mm": ranges["rain_mm"]["p95"], "usd_lkr": ranges["usd_lkr"]["current"] * 1.03}
t0 = time.perf_counter()
s = model.forecast(ORIGIN, scenario)
print(f"Scenario run in {(time.perf_counter() - t0) * 1000:.0f} ms")
print(f"Baseline {r['forecast_value']:.3f}  ->  scenario {s['forecast_value']:.3f}  "
      f"(change {s['forecast_value'] - r['forecast_value']:+.3f} million kg)")
print("Scenario warnings:", s["warnings"][1:] or "none")

try:
    model.forecast(ORIGIN, {"oni": 5.0})
except ScenarioError as err:
    print("\nRejected as expected:", err)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
for ax, (name, rg) in zip(axes, ranges.items()):
    values = np.linspace(rg["p5"], rg["p95"], 15)
    preds = [model.forecast(ORIGIN, {name: v})["forecast_value"] for v in values]
    ax.plot(values, preds, "-o", ms=3)
    ax.axvline(rg["current"], color="tab:red", ls="--", lw=1)
    ax.set_title(name, fontsize=9)
axes[0].set_ylabel("forecast (million kg)")
fig.suptitle(f"One-driver sensitivity at {ORIGIN} across the usual (p5-p95) range; red = current value", fontsize=10)
fig.tight_layout()
plt.show()

**Finding:** each driver moves the forecast by only about 0.1-0.3 million kg across its usual range, and tree splits make the response step-like. Monthly export volume in this data is driven mainly by recent momentum and seasonality. M3's simulator and rationale must report small effects honestly rather than exaggerate them.

### 7.4 Reload the saved model (what M3 will do)

In [ ]:
from tea_m3.forecasting import load_forecaster

if path is not None:
    m = load_forecaster(path)
    assert m.forecast(ORIGIN) == model.forecast(ORIGIN)
    print("Reloaded model gives identical forecasts:", m.model_version()["model_id"])
else:
    print("Model was not saved (see section 6), so there is nothing to reload.")

## Summary

- **Data:** M1's dataset cleaned to 300 months (2000-2024); every fix is in the cleaning log and `reports/dataset_qa.md`.
- **Model:** light M1-lite with 100 XGBoost trees plus a linear temporal branch, fixed blend *w* chosen on validation only.
- **Evaluation:** walk-forward on held-out 2022-2024. It beats seasonal naive and is roughly equal to persistence.
- **M3 support:** exact SHAP reconciliation, 7 business-driver groups, validated what-if scenarios in about 10 ms.
- **Limits:** stand-in model only; FX before 2014 is labelled synthetic; buyer-demand index is a proxy; no regime classifier; weak driver sensitivity.

Next for M3: the explanation module (driver chart + Jinja2 rationale), decision-prompt rules, the API and the Streamlit simulator.